# Module 1: LangGraph Basics & Prompt Engineering

**Day 4 — Agents, LangGraph & MCP**

## What you will learn
- Build a `StateGraph` from scratch
- Conditional routing with `add_conditional_edges`
- Prompt engineering: zero-shot, few-shot, chain-of-thought
- Token counting with `tiktoken`
- Context window and token limits

## 1. Simplest LangGraph — One node, one edge

A LangGraph has three ingredients: **state** (TypedDict), **nodes** (functions), and **edges** (connections).

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

class SimpleState(TypedDict):
    message: str

def greet(state: SimpleState) -> SimpleState:
    return {"message": f"Hello, {state['message']}!"}

graph = StateGraph(SimpleState)
graph.add_node("greet", greet)
graph.set_entry_point("greet")
graph.add_edge("greet", END)
app = graph.compile()

result = app.invoke({"message": "World"})
print(result)  # {'message': 'Hello, World!'}

**What happened:**
1. `StateGraph(SimpleState)` — declare state schema
2. `add_node` — register a function as a node
3. `set_entry_point` — first node to run
4. `add_edge` — connect nodes
5. `compile()` — build the runnable
6. `invoke()` — run with initial state

## 2. Conditional Routing

Route to different nodes based on state using `add_conditional_edges`.

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Literal

class NumberState(TypedDict):
    number: int
    result: str

def classify(state: NumberState) -> Literal["even", "odd"]:
    return "even" if state["number"] % 2 == 0 else "odd"

def handle_even(state): return {"result": f"{state['number']} is EVEN"}
def handle_odd(state):  return {"result": f"{state['number']} is ODD"}

g = StateGraph(NumberState)
g.add_node("router", lambda s: s)
g.add_node("even",   handle_even)
g.add_node("odd",    handle_odd)
g.set_entry_point("router")
g.add_conditional_edges("router", classify, {"even": "even", "odd": "odd"})
g.add_edge("even", END)
g.add_edge("odd",  END)
app2 = g.compile()

for n in [4, 7, 100, 13]:
    r = app2.invoke({"number": n, "result": ""})
    print(r["result"])

## 3. Prompt Engineering

| Technique | When to use | Key idea |
|-----------|-------------|----------|
| **Zero-shot** | Powerful models, simple tasks | Just describe the task |
| **Few-shot** | Complex formatting, edge cases | Show 2-5 examples |
| **Chain-of-thought** | Reasoning, math, multi-step | Add "Think step by step" |

In [ ]:
from langchain_core.prompts import PromptTemplate

# Zero-shot
zero = PromptTemplate.from_template(
    "Classify the sentiment (positive/negative/neutral):\n\n{review}"
)
print("ZERO-SHOT:")
print(zero.format(review="Shipping was slow but product is great."))
print()

In [ ]:
# Few-shot
few = PromptTemplate.from_template(
    "Classify sentiment as positive/negative/neutral.\n\n"
    "Review: Amazing quality! -> positive\n"
    "Review: Broke after one day. -> negative\n"
    "Review: Does the job. -> neutral\n\n"
    "Review: {review} ->"
)
print("FEW-SHOT:")
print(few.format(review="Battery life is decent but screen is dim."))
print()

In [ ]:
# Chain-of-thought
cot = PromptTemplate.from_template(
    "Solve this problem. Think step by step.\n\nProblem: {problem}\n\nLet's think step by step:"
)
print("CHAIN-OF-THOUGHT:")
print(cot.format(problem="If a train travels 120km in 2 hours, what is its average speed?"))

## 4. Token Counting with tiktoken

LLMs charge and limit by tokens. Always measure before sending long context.

In [ ]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o")

texts = [
    "Hello!",
    "Explain quantum computing in simple terms.",
    "Write a comprehensive analysis of the Indian economy in 2024, covering GDP growth, inflation, employment, and key sectors."
]
print(f"{'Text':<70} Tokens")
print("-" * 80)
for t in texts:
    print(f"{t[:67]:<70} {len(enc.encode(t))}")

print("\nContext window limits:")
for model, limit in [("gpt-4o-mini", 128_000), ("gpt-4o", 128_000), ("claude-3-5-sonnet", 200_000)]:
    print(f"  {model:<25}: {limit:>10,} tokens")

## 5. Using the day4 modules

In [ ]:
import sys
sys.path.insert(0, '../src')

In [ ]:
from day4.langgraph_basics import build_bmi_graph, build_review_graph, run_graph

bmi_graph = build_bmi_graph()
cases = [
    {"weight_kg": 50, "height_m": 1.70, "bmi": None, "category": None, "recommendation": None},
    {"weight_kg": 75, "height_m": 1.75, "bmi": None, "category": None, "recommendation": None},
    {"weight_kg": 110,"height_m": 1.70, "bmi": None, "category": None, "recommendation": None},
]
print("BMI Calculator:")
for c in cases:
    r = run_graph(bmi_graph, c)
    print(f"  {r['weight_kg']}kg / {r['height_m']}m → BMI {r['bmi']:.1f} ({r['category']})")

In [ ]:
from day4.prompt_engineering import zero_shot_prompt, few_shot_prompt, chain_of_thought_prompt, count_tokens, estimate_cost

text = "Explain the architecture of transformer models including attention mechanisms and positional encoding." * 2
tokens = count_tokens(text)
cost   = estimate_cost(tokens, "gpt-4o-mini")
print(f"Characters : {len(text)}")
print(f"Tokens     : {tokens}")
print(f"Cost (est) : ${cost['cost_usd']:.6f} USD on {cost['model']}")